# OceanRescue: Fishing Net & Rope Instance Segmentation Training
**Priority ALDFG Detection Model (YOLOv8-seg / YOLOv11-seg)**

This notebook trains a subsea instance segmentation model specifically for detecting:
- `0: fishing_net` (Ghost nets, gillnets)
- `1: rope_line` (Mooring lines, buoy lines, cordage)


## 1. Verify GPU Hardware

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 2. Install Ultralytics & Dependencies

In [ ]:
!pip install -q ultralytics opencv-python pycocotools matplotlib
import ultralytics
ultralytics.checks()


## 3. Unpack Dataset Package
Upload `colab_training_package.zip` into `/content/` before running this cell.

In [ ]:
import os, zipfile
zip_path = '/content/colab_training_package.zip'
extract_dir = '/content/colab_training_package'

if os.path.exists(zip_path):
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content/')
    print("Extraction complete.")
elif os.path.exists(extract_dir):
    print(f"{extract_dir} already exists.")
else:
    print("Please upload colab_training_package.zip to /content/ first!")

# Verify images and labels
tr_imgs = len(os.listdir(f"{extract_dir}/images/train"))
val_imgs = len(os.listdir(f"{extract_dir}/images/val"))
print(f"Dataset verified: {tr_imgs} train images, {val_imgs} val images.")


## 4. Train YOLOv8s-seg Model

In [ ]:
# Launch training with YOLOv8s-seg on GPU
!yolo segment train \
    data=/content/colab_training_package/data.yaml \
    model=yolov8s-seg.pt \
    epochs=50 \
    imgsz=640 \
    batch=16 \
    device=0 \
    project=OceanRescue_ALDFG \
    name=yolov8s_seg_aldfg \
    save=True


## 5. Evaluate Validation Metrics (mAP50 & mAP50-95)

In [ ]:
!yolo segment val \
    data=/content/colab_training_package/data.yaml \
    model=OceanRescue_ALDFG/yolov8s_seg_aldfg/weights/best.pt \
    imgsz=640 \
    device=0


## 6. Run Inference & Visualize Predictions

In [ ]:
!yolo segment predict \
    model=OceanRescue_ALDFG/yolov8s_seg_aldfg/weights/best.pt \
    source=/content/colab_training_package/images/val \
    save=True \
    conf=0.25 \
    project=OceanRescue_ALDFG \
    name=predictions

# Display sample prediction
import glob
from IPython.display import Image, display
pred_imgs = glob.glob('OceanRescue_ALDFG/predictions/*.jpg')
if pred_imgs:
    display(Image(filename=pred_imgs[0]))
